In [2]:
%matplotlib widget
import matplotlib.pyplot as plt
import numpy as np
from scipy.integrate import solve_ivp
from scipy.spatial.transform import Rotation as R
import plotly.graph_objects as go
from StarTracker_helper import *
import plotly.io as pio
import plotly.express as pexp
from itertools import permutations
from collections import defaultdict
from plotly.subplots import make_subplots
import ipywidgets
import torch
from PIL import Image
import os
from scipy import ndimage
from scipy.optimize import fsolve
from IPython.display import display, clear_output
import segmentation_models_pytorch as smp

if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using NVIDIA GPU (CUDA)!")
elif torch.backends.mps.is_available(): 
    device = torch.device("mps")
else:
    device = torch.device("cpu")
    print("Using CPU.")

Using NVIDIA GPU (CUDA)!


In [3]:
# Create a catalogue of randomly generated stars
rng = np.random.default_rng(seed=42)

num_stars = 250
K, pairs_sorted, _, slope, q, N_pairs, star_unit_vectors, sizes = create_catalogo(num_stars, 1, 2, 3, 10, 0.75)

In [5]:
# Orbit Parameters
Re = 6378.15e3
mu = 3.986e14
alt = 20000e3
a = Re + alt
kep0 = np.array([a, 0, np.deg2rad(0), np.deg2rad(0), np.deg2rad(0), np.deg2rad(0)])
r0, v0 = kep2car(kep0, mu)
T = 2*np.pi*np.sqrt(a**3 / mu)
tspan = np.linspace(0, T, 1000)
sol = solve_ivp(tbp, [0.0, T], np.hstack([r0, v0]), t_eval=tspan,
                    rtol=1e-10, atol=1e-12, args=(mu, 'no'))
YY = sol.y.T

threshold_multiplier = 2

image_size = 256
base_path = "test_images"
if not os.path.exists(base_path): os.makedirs(base_path)

def visualization_interactive(idx=0, fov = 60):
    plt.close('all')
    r_current = YY[idx, 0:3]
    v_current = YY[idx, 3:6]

    A0 = lvlh_frame(r_current, v_current)

    f_len = 1 / (np.tan(np.deg2rad(fov) / 2))
    sensor_noise = 0.01
    visible_stars, px, py, sensor_view = project_combined(star_unit_vectors, A0, fov, sensor_noise, image_size, sizes, YY, idx, 'vscode')

    NN = True

    if NN:
        device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
        model = USpaceNet() 
        model.load_state_dict(torch.load("USpaceNet.pth", map_location=device))
        model.eval().to(device)

        input_img = torch.from_numpy(sensor_view).unsqueeze(0).unsqueeze(0).to(device, dtype=torch.float32)
        with torch.no_grad():
            heatmap = model(input_img).cpu().squeeze() 

        found_stars = extract_all_stars(heatmap)
        n_stars_found = found_stars.shape[0]
        heatmap_np = heatmap.numpy() if hasattr(heatmap, 'numpy') else heatmap

        fig = go.Figure()

        # Heatmap
        fig.add_trace(go.Heatmap(
            z=heatmap_np, 
            colorscale='magma', 
            showscale=False, 
            hoverinfo='x+y+z'
        ))

        if n_stars_found > 0:
            fig.add_trace(go.Scatter(
                x=found_stars[:, 0] + 5, # Horizontal offset
                y=found_stars[:, 1] - 5, # Vertical offset
                mode='text',
                text=[f"S{i+1}" for i in range(n_stars_found)],
                textfont=dict(color='white', size=12),
                hoverinfo='text'
            ))

        fig.update_xaxes(showticklabels=True, constrain="domain")
        fig.update_yaxes(
            showticklabels=True, 
            autorange="reversed", 
            scaleanchor="x",     
            scaleratio=1
        )

        fig.update_layout(
            title=f"Detected: {n_stars_found} Stars",
            height=800, width=800, 
            plot_bgcolor='black',
            paper_bgcolor='black',
            font=dict(color='white'),
            margin=dict(l=10, r=10, t=50, b=10),
            showlegend=False
        )

        fig.show()

        coords_3D = np.zeros((n_stars_found, 3))

        if n_stars_found > 0:
            for i in range(n_stars_found):
                x_norm = 2.0 * (found_stars[i, 0] / image_size) - 1.0
                y_norm = 2.0 * (found_stars[i, 1] / image_size) - 1.0
                
                v = np.array([- x_norm / f_len, - y_norm / f_len, 1.0])
                v /= np.linalg.norm(v)
                coords_3D[i, :] = v
        print("\n")
    else:
        print("Standard CV mode")
        star_position_2D, output_img =  cv_star_detector(sensor_view, 2)
        n_stars_found = star_position_2D.shape[0]

        coords_3D = np.zeros((n_stars_found, 3))

        if n_stars_found > 0:
            for i in range(n_stars_found):
                x_norm = 2.0 * (star_position_2D[i, 0] / image_size) - 1.0
                y_norm = 2.0 * (star_position_2D[i, 1] / image_size) - 1.0
                
                v = np.array([- x_norm / f_len, - y_norm / f_len, 1.0])
                v /= np.linalg.norm(v)
                coords_3D[i, :] = v
                # print(f"Star {i+1} | 2D Position: [{px[i]:.3f}, {py[i]:.3f}] | 3D Vector: [{v[0]:.3f}, {v[1]:.3f}, {v[2]:.3f}]")
        print("\n")

    indices = np.random.randint(0, n_stars_found, 3)

    if n_stars_found > 2:
        s1 = coords_3D[indices[0], :]
        s2 = coords_3D[indices[1], :]
        s3 = coords_3D[indices[2], :]

        s1_norm = s1 / np.linalg.norm(s1)
        s2_norm = s2 / np.linalg.norm(s2)
        s3_norm = s3 / np.linalg.norm(s3)

        # --- 2. MEASURE GEOMETRY ---
        d1_measured = np.dot(s1_norm, s2_norm) # s1-s2
        d2_measured = np.dot(s1_norm, s3_norm) # s1-s3
        d3_measured = np.dot(s2_norm, s3_norm) # s2-s3

        # --- 3. CATALOG SEARCH ---
        tolerance = 1e-3
        start1, end1 = get_candidates(K, d1_measured, slope, q, N_pairs, tolerance)
        start2, end2 = get_candidates(K, d2_measured, slope, q, N_pairs, tolerance)
        start3, end3 = get_candidates(K, d3_measured, slope, q, N_pairs, tolerance)

        cp1 = pairs_sorted[start1 : end1]
        cp2 = pairs_sorted[start2 : end2]
        cp3 = pairs_sorted[start3 : end3]
        cp3_set = set(map(tuple, np.sort(cp3, axis=1)))

        triangles = []
        for a, b in cp1:
            for u, v in cp2:
                # Case: b is shared (s1)
                if u == b:
                    c = v
                    if tuple(sorted((a, c))) in cp3_set: triangles.append((int(a), int(b), int(c)))
                elif v == b:
                    c = u
                    if tuple(sorted((a, c))) in cp3_set: triangles.append((int(a), int(b), int(c)))
                # Case: a is shared (s1)
                if u == a:
                    c = v
                    if tuple(sorted((b, c))) in cp3_set: triangles.append((int(a), int(b), int(c)))
                elif v == a:
                    c = u
                    if tuple(sorted((b, c))) in cp3_set: triangles.append((int(a), int(b), int(c)))

        print(f"Triangles found in Catalog: {len(triangles)}")

        if triangles:
            best_val = float('inf')
            final_triangle = None
            for tri in triangles:
                d1_cat = np.dot(star_unit_vectors[tri[0]], star_unit_vectors[tri[1]])
                d2_cat = np.dot(star_unit_vectors[tri[1]], star_unit_vectors[tri[2]])
                d3_cat = np.dot(star_unit_vectors[tri[0]], star_unit_vectors[tri[2]])
                res = abs(d1_cat - d1_measured) + abs(d2_cat - d2_measured) + abs(d3_cat - d3_measured)
                if res < best_val:
                    best_val = res
                    final_triangle = tri

            # Matches s1, s2, s3 to specific IDs in the winner triangle
            d_ai = [d1_measured, d2_measured, d3_measured]
            best_perm_err = float('inf')
            correct_mapping = None

            for tri_perm in permutations(list(final_triangle)):
                # Distances based on permutation: 
                test_d1 = np.dot(star_unit_vectors[tri_perm[0]], star_unit_vectors[tri_perm[1]])
                test_d2 = np.dot(star_unit_vectors[tri_perm[0]], star_unit_vectors[tri_perm[2]])
                test_d3 = np.dot(star_unit_vectors[tri_perm[1]], star_unit_vectors[tri_perm[2]])
                
                perm_res = abs(test_d1 - d_ai[0]) + abs(test_d2 - d_ai[1]) + abs(test_d3 - d_ai[2])
                if perm_res < best_perm_err:
                    best_perm_err = perm_res
                    correct_mapping = tri_perm

            print(f"Winner Triangle: {final_triangle}")
            print(f"Verified Mapping: Star {indices[0]} -> ID {correct_mapping[0]}, Star {indices[1]} -> ID {correct_mapping[1]}, Star {indices[2]} -> ID {correct_mapping[2]}")

            actual_ids = set(np.where(visible_stars)[0])
            identified_ids = set(correct_mapping)
            matches = identified_ids.intersection(actual_ids)

            if len(matches) == 3:
                print("SUCCESS")
                # Compare Body Vectors
                for i, cat_id in enumerate(correct_mapping):
                    v_ai = [s1_norm, s2_norm, s3_norm][i]
                    v_real_body = A0 @ star_unit_vectors[cat_id]
                    err = np.rad2deg(np.arccos(np.clip(np.dot(v_ai, v_real_body), -1.0, 1.0)))
                    print(f"Star {indices[i]} (ID {cat_id}) Angular Error: {err:.6f}°")

                plt.figure(figsize=(12, 5))
                plt.title(f"Stars Selected & Triangle Identified")
                plt.imshow(heatmap, cmap='magma')
                tri_coords = found_stars[indices] 
                loop = np.vstack([tri_coords, tri_coords[0]])
                plt.plot(loop[:, 0], loop[:, 1], color='white', linestyle='-', linewidth=2, alpha=0.7)
                # plt.fill(loop[:, 0], loop[:, 1], color='cyan', alpha=0.15)
                for i in indices:
                    plt.scatter(found_stars[i, 0], found_stars[i, 1], marker='x', color='red', s=80)
                    plt.text(found_stars[i, 0] + 5, found_stars[i, 1] - 5, 
                            f"S{i}", color='white', fontsize=10, fontweight='bold')
                plt.show()

            else:
                truth_ids = sorted(int(i) for i in actual_ids)
                identified_ids = sorted(int(i) for i in identified_ids)

                print("FAILURE")
                print(f"  Identified IDs : {identified_ids}")
                print(f"  True visible IDs ({len(truth_ids)}): {truth_ids}")
        else:
            print("No triangle found. Try increasing tolerance.")

        if triangles and correct_mapping:
            obs = np.array([s1_norm, s2_norm, s3_norm])
            ref = np.array([
                star_unit_vectors[correct_mapping[0]],
                star_unit_vectors[correct_mapping[1]],
                star_unit_vectors[correct_mapping[2]]
            ])

            # Solve for Attitude
            A_est = solve_wahba(obs, ref)

            print("\n--- Attitude Estimation Results ---")
            np.set_printoptions(precision=4, suppress=True)
            print("Estimated Matrix A_est:\n", A_est)
            print("True Matrix A0:\n", A0)

            # 5. Calculate Attitude Error (Angular distance between matrices)
            # The error rotation matrix
            E = A_est @ A0.T

            trace_E = np.trace(E)
            angular_error_deg = np.rad2deg(np.arccos(np.clip((trace_E - 1) / 2, -1, 1)))

            print(f"\nFinal Pointing Accuracy: {angular_error_deg:.6f} degrees")
            plot_attitude(A0, A_est, 0.25, 0.25, 1, 0.3) # Height = Along Track, Length = Across Track, Width = Towards Nadir. To get a pencil shaped which points stars plot_attitude(A0, A_est, 0.1, 0.1, 1, 0.3)

# --- 3. Run Widget ---
ipywidgets.interact(visualization_interactive, 
                    idx=ipywidgets.IntSlider(min=0, max=len(YY)-1, step=5, value=0, description='Orbit Position'),
                    fov=ipywidgets.IntSlider(min=10, max=80, step=1, value=60, description='FOV')
                    ) 


interactive(children=(IntSlider(value=0, description='Orbit Position', max=999, step=5), IntSlider(value=60, d…

<function __main__.visualization_interactive(idx=0, fov=60)>